# E5: robustness to text perturbations (DistilBERT + BiLSTM)

Proposal 4.4-4.5 and the E5 row of section 5: single-channel fraud models (sms = Mendeley, email = Nazario_5 + Nigerian_5, phone = BothBosu), trained as in E3, are scored on their own channel's test set: clean, P1, P2, P3 (and their sub-forms), each with and without the D1 defence `normalise()` from the group repo.

Before running: Add Input -> `zyliang19/ee6405-processed` and `zyliang19/mendeley-sms`; Accelerator -> GPU T4; then Save Version -> Save & Run All (Commit).
Output: `e5_results.zip` (download from the Output tab).

In [1]:
%%writefile normalise.py
"""Text normalisation defence (D1) against character-level obfuscation.

normalise(text) undoes four kinds of obfuscation in English messages:
invisible format characters (zero-width spaces and joiners, soft hyphens,
direction marks), full-width and other compatibility letters, Unicode
look-alike letters (Cyrillic/Greek), and spaced-out letters ("v e r i f y").

It does not undo digit-for-letter swaps ("acc0unt"), because a digit map would
also change prices and phone numbers, nor dotted letters ("v.e.r.i.f.y").
Digits and symbols ("\u00bd", "\u00b2", "\u2122") are left unchanged for the same reason.

Limits: designed for English text. The look-alike map changes Cyrillic and
Greek letters, so it is not suitable for messages written in those scripts.
Adjacent spaced-out words are joined into one ("v e r i f y n o w" -> "verifynow").
"""

import re
import unicodedata

# Cyrillic and Greek letters that look like Latin letters
LOOKALIKES = str.maketrans({
    # Cyrillic
    "\u0430": "a", "\u0435": "e", "\u043e": "o", "\u0440": "p", "\u0441": "c",
    "\u0443": "y", "\u0445": "x", "\u0456": "i", "\u0458": "j", "\u0455": "s",
    "\u0501": "d", "\u04cf": "l", "\u04bb": "h", "\u051d": "w", "\u051b": "q",
    "\u0475": "v",
    "\u043a": "k",                                                    # lowercase ka
    "\u0410": "A", "\u0412": "B", "\u0415": "E", "\u041a": "K", "\u041c": "M",
    "\u041d": "H", "\u041e": "O", "\u0420": "P", "\u0421": "C", "\u0422": "T",
    "\u0423": "Y", "\u0425": "X", "\u0406": "I", "\u0408": "J", "\u0405": "S",
    # Latin-extended
    "\u0131": "i", "\u0261": "g", "\u0251": "a",
    # Greek
    "\u03b1": "a", "\u03bf": "o", "\u03c1": "p", "\u03bd": "v", "\u03b9": "i",
    "\u03ba": "k", "\u03c5": "u", "\u03f2": "c", "\u03f3": "j",
    "\u0391": "A", "\u0392": "B", "\u0395": "E", "\u0396": "Z", "\u0397": "H",
    "\u0399": "I", "\u039a": "K", "\u039c": "M", "\u039d": "N", "\u039f": "O",
    "\u03a1": "P", "\u03a4": "T", "\u03a5": "Y", "\u03a7": "X",
})

# Four or more single letters separated by single spaces: "v e r i f y".
# Three ("I a m", "A B C") are left alone, as they are often real words or initials.
SPACED_LETTERS_RE = re.compile(r"(?<![A-Za-z])[A-Za-z](?: [A-Za-z]){3,}(?![A-Za-z])")


def normalise_with_changes(text: str) -> tuple[str, dict]:
    """Like normalise(), and also count what each step changed.

    The counts are characters removed (invisible_chars), characters replaced
    (compat_chars, lookalike_letters) and spaced-out letter runs joined
    (spaced_letters). All zeros means the text was not changed.
    """
    changes = {}
    # Invisible format characters (Unicode category Cf), e.g. U+200B, U+00AD, U+200E
    kept = [c for c in text if unicodedata.category(c) != "Cf"]
    changes["invisible_chars"] = len(text) - len(kept)
    # NFKC on letters and spaces only: full-width and math letters become ASCII,
    # a no-break space becomes a space; digits and symbols are not touched
    compat = [
        unicodedata.normalize("NFKC", c) if unicodedata.category(c)[0] in "LZ" else c
        for c in kept
    ]
    changes["compat_chars"] = sum(a != b for a, b in zip(kept, compat))
    text = "".join(compat)
    changes["lookalike_letters"] = sum(ord(c) in LOOKALIKES for c in text)
    text = text.translate(LOOKALIKES)
    text, changes["spaced_letters"] = SPACED_LETTERS_RE.subn(
        lambda m: m.group(0).replace(" ", ""), text
    )
    return text, changes


def normalise(text: str) -> str:
    """Return text with invisible characters, look-alike letters and spaced-out letters undone."""
    return normalise_with_changes(text)[0]


Writing normalise.py


In [2]:
%%writefile perturb.py
"""Rule-based text perturbations for the robustness test E5 (EE6405 proposal v2, 4.4).

Three types of surface change, each applied to 30% of the eligible words of a message:
  P1  character substitution:      account -> acc0unt, verify -> ver1fy
  P2  inserted spaces or dots:     verify  -> v e r i f y  or  v.e.r.i.f.y
  P3  Unicode look-alikes or a zero-width character:
                                   account -> аccount (Cyrillic a), ver​ify
P2 and P3 pick one of their two forms per word at random. Each form can also be used on its
own (P2_space, P2_dot, P3_lookalike, P3_zero_width), because the D1 defence
(src/normalise.py) undoes P2_space, P3_lookalike and P3_zero_width but not P2_dot or P1
(proposal 4.5), and the DistilBERT tokenizer may already drop zero-width characters.

Eligible words are runs of 4 or more ASCII letters that are not English stop words
(NLTK's list) and are not part of a URL or email address. The number of words changed is
round(0.3 x eligible) (Python rounding), and at least 1 when the message has any eligible word.
P1 and P3_lookalike change one letter per word, and only words with a letter they can
change are picked. The rules change only the surface form, so the true label is kept.

Results are reproducible: pass the same seed and message key (for example the sample_id)
to get the same change.

Usage:
    from perturb import perturb, perturb_with_changes, KINDS
    perturb("Please verify your account now", "P1", key="msg_1")
    python perturb.py                  # prints one example of every kind

Outputs: the changed text; perturb_with_changes() also returns the number of eligible
and changed words.
"""
from __future__ import annotations

import random
import re
from typing import TYPE_CHECKING, Callable, Optional

if TYPE_CHECKING:
    import pandas as pd

# --------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------
RATE = 0.3
MIN_LEN = 4
KINDS = ["P1", "P2", "P2_space", "P2_dot", "P3", "P3_lookalike", "P3_zero_width"]

# NLTK's English stop-word list (179 words), copied here to avoid the dependency. Only words of
# 4+ letters matter. scikit-learn's list is not used: it contains scam words such as "call" and "bill".
STOP_WORDS = frozenset("""
i me my myself we our ours ourselves you you're you've you'll you'd your yours yourself yourselves
he him his himself she she's her hers herself it it's its itself they them their theirs themselves
what which who whom this that that'll these those am is are was were be been being have has had
having do does did doing a an the and but if or because as until while of at by for with about
against between into through during before after above below to from up down in out on off over
under again further then once here there when where why how all any both each few more most other
some such no nor not only own same so than too very s t can will just don don't should should've
now d ll m o re ve y ain aren aren't couldn couldn't didn didn't doesn doesn't hadn hadn't hasn
hasn't haven haven't isn isn't ma mightn mightn't mustn mustn't needn needn't shan shan't shouldn
shouldn't wasn wasn't weren weren't won won't wouldn wouldn't
""".split())

WORD_RE = re.compile(r"[A-Za-z]+")
# URLs and email addresses are left untouched: changing them would change where a link goes
# (bounded repeats keep the regex linear on very long emails, e.g. base64 attachments)
PROTECTED_RE = re.compile(r"(?:https?://|www\.)\S+|[\w.+-]{1,64}@[\w-]{1,63}(?:\.[\w-]{1,63})+"
                          r"|\b[\w-]{1,63}\.(?:com|net|org|co|uk|xyz|info|biz)\b\S*", re.IGNORECASE)

# P1: letter -> digit or symbol that reads the same
P1_MAP = {"a": "@", "e": "3", "i": "1", "o": "0", "s": "5", "l": "1", "t": "7", "b": "8", "g": "9",
          "A": "4", "E": "3", "I": "1", "O": "0", "S": "5", "T": "7", "B": "8"}
# P3: Latin letter -> Cyrillic look-alike (all of them are undone by src/normalise.py)
P3_MAP = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "y": "у",
          "x": "х", "i": "і", "j": "ј", "s": "ѕ", "h": "һ",
          "A": "А", "B": "В", "E": "Е", "K": "К", "M": "М", "H": "Н",
          "O": "О", "P": "Р", "C": "С", "T": "Т", "X": "Х"}
ZERO_WIDTH = "​"


# --------------------------------------------------------------------------
# Word-level changes
# --------------------------------------------------------------------------
def _swap_one(word: str, mapping: dict[str, str], rng: random.Random) -> str:
    """Replace one random letter of word that has an entry in mapping."""
    pos = [i for i, c in enumerate(word) if c in mapping]
    i = rng.choice(pos)
    return word[:i] + mapping[word[i]] + word[i + 1:]


def p1(word: str, rng: random.Random) -> str:
    """P1: one letter becomes a digit or symbol (acc0unt)."""
    return _swap_one(word, P1_MAP, rng)


def p2_space(word: str, rng: random.Random) -> str:
    """P2, spaced form: v e r i f y."""
    return " ".join(word)


def p2_dot(word: str, rng: random.Random) -> str:
    """P2, dotted form: v.e.r.i.f.y."""
    return ".".join(word)


def p3_lookalike(word: str, rng: random.Random) -> str:
    """P3, look-alike form: one Latin letter becomes its Cyrillic twin."""
    return _swap_one(word, P3_MAP, rng)


def p3_zero_width(word: str, rng: random.Random) -> str:
    """P3, zero-width form: a zero-width space inside the word."""
    i = rng.randint(1, len(word) - 1)
    return word[:i] + ZERO_WIDTH + word[i:]


WordFn = Callable[[str, random.Random], str]
SINGLE: dict[str, WordFn] = {"P1": p1, "P2_space": p2_space, "P2_dot": p2_dot,
                             "P3_lookalike": p3_lookalike, "P3_zero_width": p3_zero_width}
MIXED: dict[str, list[WordFn]] = {"P2": [p2_space, p2_dot], "P3": [p3_lookalike, p3_zero_width]}


def _can_change(word: str, fn: WordFn) -> bool:
    """Whether fn has something to change in word (P1 and P3_lookalike need a mapped letter)."""
    if fn is p1:
        return any(c in P1_MAP for c in word)
    if fn is p3_lookalike:
        return any(c in P3_MAP for c in word)
    return True


# --------------------------------------------------------------------------
# Message-level perturbation
# --------------------------------------------------------------------------
def eligible_spans(text: str) -> list[tuple[int, int]]:
    """Character spans of the eligible words: 4+ letters, not a stop word, not in a URL or email."""
    protected = [m.span() for m in PROTECTED_RE.finditer(text)]   # sorted, non-overlapping
    spans, j = [], 0
    for m in WORD_RE.finditer(text):
        while j < len(protected) and protected[j][1] <= m.start():
            j += 1
        if j < len(protected) and protected[j][0] <= m.start():
            continue                                               # inside a URL or email
        w = m.group(0)
        if len(w) >= MIN_LEN and w.lower() not in STOP_WORDS:
            spans.append(m.span())
    return spans


def perturb_with_changes(text: str, kind: str, rate: float = RATE, seed: int = 0,
                         key: Optional[str] = None) -> tuple[str, int, int]:
    """Apply perturbation kind to round(rate x eligible) words (at least 1 if any are eligible).

    key (for example the sample_id) and seed make the result reproducible.
    Returns (changed text, number of eligible words, number of changed words).
    """
    if kind not in KINDS:
        raise ValueError(f"unknown kind {kind!r}; choose from {KINDS}")
    rng = random.Random(f"{seed}|{kind}|{key if key is not None else text}")
    spans = eligible_spans(text)
    fns = MIXED.get(kind) or [SINGLE[kind]]
    # draw the form per word first, then keep only the words that form can change
    choices = [(s, rng.choice(fns)) for s in spans]
    choices = [(s, fn) for s, fn in choices if _can_change(text[s[0]:s[1]], fn)]
    n_change = min(len(choices), max(1, round(rate * len(spans)))) if spans else 0
    picked = sorted(rng.sample(choices, n_change), key=lambda c: c[0][0])
    out, last = [], 0
    for (a, b), fn in picked:
        out.append(text[last:a])
        out.append(fn(text[a:b], rng))
        last = b
    out.append(text[last:])
    return "".join(out), len(spans), n_change


def perturb(text: str, kind: str, rate: float = RATE, seed: int = 0, key: Optional[str] = None) -> str:
    """Return text with perturbation kind applied (see perturb_with_changes)."""
    return perturb_with_changes(text, kind, rate, seed, key)[0]


# --------------------------------------------------------------------------
# E5 test sets
# --------------------------------------------------------------------------
def e5_test_sets(test: "pd.DataFrame", seed: int = 0, text_col: str = "text"
                 ) -> tuple[dict[str, "pd.DataFrame"], "pd.DataFrame"]:
    """Build the E5 evaluation sets from one clean test set (proposal 5, table row E5).

    Returns ({name: DataFrame}, log). Names are "clean", every kind in KINDS, and each of
    these with "+D1" (normalise() applied after the change; "clean+D1" checks that D1 does
    not create false alarms on clean text). The log has one row per message and kind with
    sample_id, kind, the original and changed text and the eligible / changed word counts.
    """
    import pandas as pd
    try:
        from normalise import normalise
    except ImportError:          # inside the group repo
        from src.normalise import normalise

    sets = {"clean": test}
    log = []
    for kind in KINDS:
        out = [perturb_with_changes(t, kind, seed=seed, key=str(i))
               for t, i in zip(test[text_col].fillna(""), test["id"])]
        d = test.copy()
        d[text_col] = [o[0] for o in out]
        sets[kind] = d
        log.append(pd.DataFrame({"sample_id": test["id"].values, "kind": kind,
                                 "original": test[text_col].values, "text": d[text_col].values,
                                 "n_eligible": [o[1] for o in out], "n_changed": [o[2] for o in out]}))
    for name in list(sets):
        d = sets[name].copy()
        d[text_col] = d[text_col].fillna("").map(normalise)
        sets[f"{name}+D1"] = d
    return sets, pd.concat(log, ignore_index=True)


if __name__ == "__main__":
    example = "URGENT: your bank account has been suspended. Verify now at http://secure-bank.xyz"
    for k in KINDS:
        t, n_elig, n_chg = perturb_with_changes(example, k, key="demo")
        print(f"{k:14s} {n_chg}/{n_elig}  {t!r}")


Writing perturb.py


In [3]:
%%writefile test_perturb.py
"""pytest checks for the E5 perturbation rules (proposal 4.4 and 4.7).

Usage:
    pytest test_perturb.py      # from the folder that holds perturb.py and normalise.py
"""
from __future__ import annotations

import pytest

from perturb import KINDS, ZERO_WIDTH, eligible_spans, perturb, perturb_with_changes

try:
    from normalise import normalise
except ImportError:          # inside the group repo
    from src.normalise import normalise

MSG = "URGENT: your bank account has been suspended. Verify now at http://secure-bank.xyz"


def test_eligible_words() -> None:
    """Only 4+ letter non-stop words outside URLs are eligible."""
    words = [MSG[a:b] for a, b in eligible_spans(MSG)]
    assert words == ["URGENT", "bank", "account", "suspended", "Verify"]


@pytest.mark.parametrize("kind", KINDS)
def test_rate_and_reproducible(kind: str) -> None:
    """30% of 5 eligible words -> 2 changed; same seed and key give the same text."""
    t1, n_elig, n_chg = perturb_with_changes(MSG, kind, key="k")
    assert (n_elig, n_chg) == (5, 2)
    assert t1 != MSG
    assert perturb(MSG, kind, key="k") == t1


@pytest.mark.parametrize("kind", KINDS)
def test_url_and_short_words_untouched(kind: str) -> None:
    """The URL and the short / stop words keep their exact form."""
    out = perturb(MSG, kind, key="k")
    assert out.endswith("http://secure-bank.xyz")
    assert " your " in out and " now at " in out


def test_at_least_one_word() -> None:
    """A message with one eligible word still gets one change."""
    assert perturb("call me", "P2_dot") == "c.a.l.l me"


def test_no_eligible_words() -> None:
    """Messages without eligible words are returned unchanged."""
    assert perturb("ok see you at 5", "P1") == "ok see you at 5"


def test_p1_changes_one_letter() -> None:
    """P1 swaps exactly one letter of the word for a digit or symbol."""
    out = perturb("verify", "P1")
    assert len(out) == 6 and sum(a != b for a, b in zip(out, "verify")) == 1


@pytest.mark.parametrize("kind", ["P2_space", "P3_lookalike", "P3_zero_width", "P3"])
def test_normalise_undoes(kind: str) -> None:
    """D1 restores spaced-out letters, look-alikes and zero-width characters (4.5)."""
    assert normalise(perturb("Please verify", kind)) == "Please verify"


@pytest.mark.parametrize("kind", ["P1", "P2_dot"])
def test_normalise_does_not_undo(kind: str) -> None:
    """D1 leaves digit swaps and dotted words as they are (4.5)."""
    assert normalise(perturb("Please verify", kind)) != "Please verify"


def test_zero_width_inside_word() -> None:
    """The zero-width space goes inside the word, not at its ends."""
    out = perturb("verify", "P3_zero_width")
    assert ZERO_WIDTH in out[1:-1] and out.replace(ZERO_WIDTH, "") == "verify"


def test_unknown_kind() -> None:
    """An unknown kind is an error, not a silent no-op."""
    with pytest.raises(ValueError):
        perturb(MSG, "P9")


Writing test_perturb.py


In [4]:
%%writefile e3_e4_transfer.py
"""E3 (cross-channel) and E4 (cross-corpus email) transfer for DistilBERT, EE6405 proposal v2.

E3 (4.4): binary fraud detection on label_fraud (1 = fraud, 0 = legitimate; marketing
spam and insurance/sales calls are NA and dropped). Channels: sms = mendeley,
email = email_fraud, phone = bothbosu. Training sources: the 3 single channels and the
3 pairs; every model is evaluated on the test set of every channel (same channel =
in-domain, other channels = transfer).
E4 (4.4): binary spam detection on label_spam, CEAS_08 -> SpamAssassin and the reverse,
once on the raw text and once on text_stripped (header/format traces removed). When
CEAS_08 is the target on split_seed0, results are also reported without the CNN
"Daily Top 10" campaign, which appears only in that test set.
E3M (size-matched control): each two-channel E3 run is repeated with its training set
randomly subsampled (random_state = split seed) to the size of the larger of its two
channels, to separate the effect of a second channel from the effect of more data.
E5 (4.4-4.5): the single-channel E3 models are trained again in the same way and scored on
their own channel's test set, clean and after each perturbation of perturb.py (P1, P2, P3 and
their sub-forms), each also after the D1 defence normalise() (src/normalise.py). The changed
texts are saved as e5_texts_<channel>_seed<k>.csv.gz so other models can use the same ones.
All experiments call remove_overlap (same logic as src/splits.py) to drop source rows
that duplicate any target test set, train on the source train split only (source val
is just recorded), and reuse the Mendeley E2 setting (lr 3e-5, 3 epochs, class weights,
max 512 tokens). A failing run does not stop the others; tracebacks go to errors.txt.

Inputs: mendeley / email_fraud / bothbosu / ceas_08 / spamassassin as .csv or .csv.gz in
any folder under DATA_DIR (default /kaggle/input).

Usage (Kaggle: attach the processed datasets, GPU on, then Save & Run All):
    python e3_e4_transfer.py
    EXPERIMENTS=E4 SEEDS=0 python e3_e4_transfer.py    # one experiment, one split seed
    EXPERIMENTS=E3M python e3_e4_transfer.py          # E3 size-matched control only
    EXPERIMENTS=E5 python e3_e4_transfer.py           # E5 robustness (needs perturb.py, normalise.py)

Outputs: results_e3_e4/ (e3_e4_summary.csv, e3_e4_mean_std.csv, metrics_<run>.json,
pred_<run>_<eval set>.csv, e5_texts_*.csv.gz for E5, env.json, errors.txt if any).
Models are not saved.
"""
from __future__ import annotations

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # one T4 only: batch 16 is the real batch

import glob
import json
import platform
import re
import time
import traceback
from typing import Any, Callable

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_recall_fscore_support)
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

# --------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")
OUT = os.environ.get("OUT", "/kaggle/working")
DRY_RUN = int(os.environ.get("DRY_RUN", "0"))   # > 0: use a few rows per split to test the code
MODEL = os.environ.get("MODEL", "distilbert/distilbert-base-uncased")
EXPERIMENTS = os.environ.get("EXPERIMENTS", "E3,E4").split(",")
SEEDS = [int(s) for s in os.environ.get("SEEDS", "0,1,2").split(",")]
LR, EPOCHS, BATCH, SEED, MAX_LEN = 3e-5, 3, 16, 42, 512
if DRY_RUN:
    EPOCHS = 1
FP16 = torch.cuda.is_available()

RES = f"{OUT}/results_e3_e4"
os.makedirs(RES, exist_ok=True)
tok = AutoTokenizer.from_pretrained(MODEL)
print("device:", torch.cuda.get_device_name(0) if FP16 else "cpu", flush=True)


def dump(obj: Any, path: str) -> None:
    """Write obj as indented JSON, converting numpy scalars to Python numbers."""
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False,
                  default=lambda o: o.item() if hasattr(o, "item") else str(o))


# --------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------
def load(name: str) -> pd.DataFrame:
    """Load <name>.csv(.gz) from DATA_DIR with nullable integer labels and no NaN text."""
    hits = [p for p in glob.glob(f"{DATA_DIR}/**/{name}.csv*", recursive=True) if os.path.isfile(p)]
    if not hits:
        raise FileNotFoundError(f"{name}.csv / {name}.csv.gz not found under {DATA_DIR}")
    df = pd.read_csv(hits[0], dtype={"label_fraud": "Int64", "label_spam": "Int64"})
    df["text"] = df["text"].fillna("")
    if "text_stripped" in df:
        df["text_stripped"] = df["text_stripped"].fillna("")
    df["template"] = df["template"].fillna("")
    return df


# Same de-duplication logic as the group's src/data.py and src/splits.py
SPACES_RE = re.compile(r"\s+")


def dedup_key(text: str) -> str:
    """Lower-cased text with whitespace collapsed, used to find exact duplicates."""
    return SPACES_RE.sub(" ", text).strip().lower()


def remove_overlap(source: pd.DataFrame, target: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """Drop source rows whose text or template also appears in target (4.4, leakage control).

    Returns the filtered source and the counts removed by exact text, by template and in total.
    """
    exact = source["text"].map(dedup_key).isin(set(target["text"].map(dedup_key)))
    template = source["template"].isin(set(target["template"]))
    counts = {"exact_text": int(exact.sum()), "template": int(template.sum()),
              "removed": int((exact | template).sum())}
    return source[~(exact | template)], counts


def to_ds(d: pd.DataFrame, text_col: str, label_col: str) -> Dataset:
    """Tokenize text_col (truncated to MAX_LEN tokens) into a HF Dataset with integer labels."""
    ds = Dataset.from_dict({"text": d[text_col].tolist(), "label": d[label_col].astype(int).tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])


# --------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------
class WeightedTrainer(Trainer):
    """HF Trainer with class-weighted cross-entropy (4.2: handles class imbalance)."""

    def __init__(self, *a: Any, class_weights: torch.Tensor | None = None, **k: Any) -> None:
        super().__init__(*a, **k)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """Cross-entropy on the logits, weighted when class_weights is set."""
        labels = inputs.pop("labels")
        out = model(**inputs)
        w = None if self.class_weights is None else self.class_weights.to(out.logits.device)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=w)
        return (loss, out) if return_outputs else loss


# --------------------------------------------------------------------------
# Evaluation
# --------------------------------------------------------------------------
def metrics(y: np.ndarray, prob: np.ndarray, labels: list[str]) -> dict[str, Any]:
    """Binary metrics from proposal 4.5 (class 1 is positive): accuracy, macro-F1,
    per-class scores, positive recall / false-positive rate / PR-AUC."""
    pred = prob.argmax(1)
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    neg = y != 1
    return {
        "n": int(len(y)), "n_positive": int((y == 1).sum()),
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro", labels=[0, 1], zero_division=0),
        "per_class": {labels[i]: {"precision": p[i], "recall": r[i], "f1": f[i], "support": int(s[i])}
                      for i in range(2)},
        "positive_recall": r[1],
        "positive_false_positive_rate": float(((pred == 1) & neg).sum() / max(neg.sum(), 1)),
        "positive_pr_auc": average_precision_score((y == 1).astype(int), prob[:, 1])
                           if 0 < (y == 1).sum() < len(y) else None,
        "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
    }


# --------------------------------------------------------------------------
# Training and evaluation of one run
# --------------------------------------------------------------------------
def train_and_eval(exp: str, tag: str, train_d: pd.DataFrame, val_d: pd.DataFrame,
                   tests: dict[str, pd.DataFrame], labels: list[str], text_col: str,
                   label_col: str, extra: dict[str, Any]) -> list[dict[str, Any]]:
    """Train once on train_d and evaluate on source val and every test set in tests
    (name -> DataFrame). Writes metrics and predictions; returns the summary rows."""
    if DRY_RUN:
        train_d = train_d.groupby(label_col, group_keys=False).head(DRY_RUN // 2)
        val_d = val_d.head(DRY_RUN)
        tests = {k: v.groupby(label_col, group_keys=False).head(DRY_RUN // 2) for k, v in tests.items()}
    set_seed(SEED)
    tr = to_ds(train_d, text_col, label_col)
    counts = np.bincount(tr["label"], minlength=2)
    w = torch.tensor(len(tr) / (2 * np.maximum(counts, 1)), dtype=torch.float)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=2, id2label=dict(enumerate(labels)), label2id={l: i for i, l in enumerate(labels)},
        ignore_mismatched_sizes=bool(DRY_RUN))
    args = TrainingArguments(output_dir=f"{OUT}/tmp", num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
        learning_rate=LR, weight_decay=0.01, fp16=FP16, seed=SEED, report_to="none",
        eval_strategy="no", save_strategy="no", logging_strategy="epoch")
    trainer = WeightedTrainer(model=model, args=args, train_dataset=tr,
                              data_collator=DataCollatorWithPadding(tok), class_weights=w)
    t0 = time.time()
    trainer.train()
    train_s = time.time() - t0

    detail = {"experiment": exp, "run": tag, **extra, "train_seconds": round(train_s, 1),
              "train_counts": dict(zip(labels, counts.tolist())), "class_weights": w.tolist(),
              "eval": {}}
    rows = []
    for name, d in {"source_val": val_d, **tests}.items():
        if len(d) == 0:
            continue
        p = trainer.predict(to_ds(d, text_col, label_col))
        prob = torch.softmax(torch.tensor(p.predictions, dtype=torch.float32), -1).numpy()
        y = p.label_ids
        m = metrics(y, prob, labels)
        detail["eval"][name] = m
        pd.DataFrame({"sample_id": d["id"].values, "eval_set": name,
                      "true_label": [labels[i] for i in y],
                      "pred_label": [labels[i] for i in prob.argmax(1)],
                      f"score_{labels[1]}": prob[:, 1]}
                     ).to_csv(f"{RES}/pred_{tag}_{name}.csv", index=False)
        rows.append({"experiment": exp, "run": tag, **{k: v for k, v in extra.items() if not isinstance(v, dict)},
                     "eval_set": name, "n": m["n"], "macro_f1": m["macro_f1"], "accuracy": m["accuracy"],
                     "pos_recall": m["positive_recall"], "pos_fpr": m["positive_false_positive_rate"],
                     "pos_pr_auc": m["positive_pr_auc"], "train_seconds": round(train_s, 1)})
        # CEAS_08 split_seed0: also report results without the CNN Daily Top 10 campaign
        if "ceas_08" in name and "seed0" in tag:
            keep = ~d["template"].str.startswith("thedailytop").values
            m2 = metrics(y[keep], prob[keep], labels)
            detail["eval"][name + "_without_dailytop"] = m2
            rows.append({**rows[-1], "eval_set": name + "_without_dailytop", "n": m2["n"],
                         "macro_f1": m2["macro_f1"], "accuracy": m2["accuracy"],
                         "pos_recall": m2["positive_recall"], "pos_fpr": m2["positive_false_positive_rate"],
                         "pos_pr_auc": m2["positive_pr_auc"]})
    dump(detail, f"{RES}/metrics_{tag}.json")
    del trainer.model
    del trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rows


summary: list[dict[str, Any]] = []


def record(rows: list[dict[str, Any]]) -> None:
    """Append result rows to the summary CSV (rewritten after every run) and print them."""
    summary.extend(rows)
    pd.DataFrame(summary).to_csv(f"{RES}/e3_e4_summary.csv", index=False)
    for r in rows:
        print("[result]", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}, flush=True)


def safe(label: str, fn: Callable[[], None]) -> None:
    """Run fn; on error, log the traceback to errors.txt and continue with the next run."""
    try:
        fn()
    except Exception:
        msg = f"### {label}\n{traceback.format_exc()}\n"
        print(msg, flush=True)
        with open(f"{RES}/errors.txt", "a") as f:
            f.write(msg)


# --------------------------------------------------------------------------
# E3: cross-channel transfer
# --------------------------------------------------------------------------
if "E3" in EXPERIMENTS or "E3M" in EXPERIMENTS or "E5" in EXPERIMENTS:
    CH = {"sms": "mendeley", "email": "email_fraud", "phone": "bothbosu"}
    data = {}
    for ch, name in CH.items():
        d = load(name)
        data[ch] = d[d["label_fraud"].notna()].reset_index(drop=True)
        print(f"E3 {ch} ({name}):", data[ch]["label_fraud"].value_counts().to_dict(), flush=True)
    SOURCES = [("sms",), ("email",), ("phone",), ("sms", "email"), ("sms", "phone"), ("email", "phone")]
    # E3M (size-matched control, 5 Experiments table): the two-channel runs again, with the
    # training set randomly subsampled to the size of the larger of its two channels.
    RUNS = ([("E3", s) for s in SOURCES] if "E3" in EXPERIMENTS else []) + \
           ([("E3M", s) for s in SOURCES if len(s) == 2] if "E3M" in EXPERIMENTS else []) + \
           ([("E5", s) for s in SOURCES if len(s) == 1] if "E5" in EXPERIMENTS else [])
    # E5 (robustness, 4.4-4.5): the single-channel models of E3, trained the same way, scored on
    # their own channel's test set: clean, every perturbation kind, and each again after D1.
    if "E5" in EXPERIMENTS:
        from perturb import e5_test_sets
    for k in SEEDS:
        sc = f"split_seed{k}"
        tests = {f"test_{ch}": d[d[sc] == "test"] for ch, d in data.items()}
        for exp, src in RUNS:
            def run(exp=exp, src=src, k=k, sc=sc, tests=tests):
                """Train on the source channel(s) of this split seed and test on all channels."""
                tag = f"{exp}_{'+'.join(src)}_seed{k}"
                tr_parts, va_parts, overlap = [], [], {}
                for ch in src:
                    d = data[ch]
                    tr, va = d[d[sc] == "train"], d[d[sc] == "val"]
                    for tch, td in tests.items():          # drop source rows that duplicate any test set
                        if tch == f"test_{ch}":
                            continue                        # same-channel split is already template-grouped
                        tr, c1 = remove_overlap(tr, td)
                        va, c2 = remove_overlap(va, td)
                        overlap[f"{ch}->{tch}"] = {"train": c1, "val": c2}
                    tr_parts.append(tr)
                    va_parts.append(va)
                train = pd.concat(tr_parts)
                extra = {"source": "+".join(src), "seed": k, "overlap_removed": overlap}
                if exp == "E3M":
                    n = max(len(t) for t in tr_parts)
                    train = pd.concat(tr_parts, keys=list(src)).sample(n=n, random_state=k)
                    kept = train.index.get_level_values(0).value_counts()
                    extra["n_train"] = n
                    extra["size_matched"] = {"n_train": n, "per_channel": {ch: int(kept.get(ch, 0)) for ch in src}}
                eval_sets = tests
                if exp == "E5":
                    ch = src[0]
                    variants, log = e5_test_sets(tests[f"test_{ch}"], seed=0)
                    log.to_csv(f"{RES}/e5_texts_{ch}_seed{k}.csv.gz", index=False)
                    eval_sets = {f"test_{ch}_{name}": d for name, d in variants.items()}
                record(train_and_eval(exp, tag, train, pd.concat(va_parts), eval_sets,
                                      ["legitimate", "fraud"], "text", "label_fraud", extra))
            safe(f"{exp} {src} seed{k}", run)

# --------------------------------------------------------------------------
# E4: cross-corpus email transfer (CEAS_08 <-> SpamAssassin, raw vs stripped)
# --------------------------------------------------------------------------
if "E4" in EXPERIMENTS:
    corp = {n: load(n) for n in ["ceas_08", "spamassassin"]}
    for n, d in corp.items():
        corp[n] = d[d["label_spam"].notna()].reset_index(drop=True)
        print(f"E4 {n}:", corp[n]["label_spam"].value_counts().to_dict(), flush=True)
    for k in SEEDS:
        sc = f"split_seed{k}"
        for src, tgt in [("spamassassin", "ceas_08"), ("ceas_08", "spamassassin")]:
            for text_col in ["text", "text_stripped"]:
                def run(src=src, tgt=tgt, text_col=text_col, k=k, sc=sc):
                    """Train on src and test on both the src and tgt test sets."""
                    s, t = corp[src], corp[tgt]
                    tgt_test = t[t[sc] == "test"]
                    tr, c1 = remove_overlap(s[s[sc] == "train"], tgt_test)
                    va, c2 = remove_overlap(s[s[sc] == "val"], tgt_test)
                    tests = {f"test_{src}": s[s[sc] == "test"], f"test_{tgt}": tgt_test}
                    variant = "stripped" if text_col == "text_stripped" else "raw"
                    tag = f"E4_{src}_to_{tgt}_{variant}_seed{k}"
                    extra = {"source": src, "target": tgt, "text": variant, "seed": k,
                             "overlap_removed": {"train": c1, "val": c2}}
                    record(train_and_eval("E4", tag, tr, va, tests, ["legitimate", "spam"],
                                          text_col, "label_spam", extra))
                safe(f"E4 {src}->{tgt} {text_col} seed{k}", run)

# --------------------------------------------------------------------------
# Summary
# --------------------------------------------------------------------------
s = pd.DataFrame(summary)
if len(s):
    keys = [c for c in ["experiment", "source", "target", "text", "eval_set"] if c in s]
    agg = s.groupby(keys, dropna=False)[["macro_f1", "pos_recall", "pos_fpr", "pos_pr_auc"]] \
           .agg(["mean", "std", "count"]).round(4)
    agg.to_csv(f"{RES}/e3_e4_mean_std.csv")
    print(agg.to_string())
dump({"lr": LR, "epochs": EPOCHS, "batch_size": BATCH, "seed": SEED, "max_length": MAX_LEN,
      "fp16": FP16, "seeds": SEEDS, "python": platform.python_version(), "torch": torch.__version__,
      "transformers": transformers.__version__,
      "device": torch.cuda.get_device_name(0) if FP16 else "cpu"}, f"{RES}/env.json")
print("All done")


Writing e3_e4_transfer.py


In [5]:
%%writefile bilstm_experiments.py
"""BiLSTM scam classifier (Keras, trainable embedding), following the EE6405 proposal v2.

Runs E2 (tuning on Mendeley), E1 (every dataset), E3 (cross-channel), E4
(cross-corpus), E3M (E3 two-channel runs with the training set subsampled to the size of
the larger channel) and E5 (robustness to the perturbations of perturb.py, with and without
the D1 defence normalise()) with the shared splits (split_seed0/1/2), so results are
directly comparable with the DistilBERT runs.

Model (4.2): string input -> TextVectorization (adapted on the training set only,
20,000 words) -> Embedding(128, mask_zero) -> Bidirectional LSTM -> Dropout -> softmax.
Training: Adam, batch 64, up to 10 epochs with early stopping on validation loss
(patience 2, best weights restored), class weights from the training set (4.2).
Tuning (4.3): lr {1e-3, 3e-4} x hidden {64, 128} x dropout {0.2, 0.5} on Mendeley
split_seed0, chosen by validation macro-F1, plus one run without class weights.
E3/E4 call remove_overlap (same logic as src/splits.py) before training.
Sequence length: 100 words for SMS, 400 for email and phone (longer texts are cut).

Usage (Kaggle: attach the processed datasets, GPU on, then Save & Run All):
    python bilstm_experiments.py
    EXPERIMENTS=E1,E3 BEST=1e-3,128,0.5 python bilstm_experiments.py   # skip tuning
    EXPERIMENTS=E3M BEST=1e-3,64,0.2 python bilstm_experiments.py      # E3 size-matched control only
    EXPERIMENTS=E5 BEST=1e-3,64,0.2 python bilstm_experiments.py       # E5 robustness (needs perturb.py)

Outputs: results_bilstm/ (bilstm_summary.csv, bilstm_mean_std.csv, e2_tuning_val.csv,
metrics_<run>.json, pred_<run>_<eval set>.csv) and models_bilstm/ (Mendeley-tuned
E1 models for seed 0 and the single-channel fraud models from E3, kept for E5).
"""
from __future__ import annotations

import glob
import json
import os
import platform
import re
import time
import traceback
from typing import Callable, Optional

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_recall_fscore_support)

# --------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")
OUT = os.environ.get("OUT", "/kaggle/working")
DRY_RUN = int(os.environ.get("DRY_RUN", "0"))        # > 0: use a few rows per split to test the code
EXPERIMENTS = os.environ.get("EXPERIMENTS", "E2,E1,E3,E4").split(",")
SEEDS = [int(s) for s in os.environ.get("SEEDS", "0,1,2").split(",")]

VOCAB, EMB, BATCH, MAX_EPOCHS, SEED = 20000, 128, 64, 10, 42
if DRY_RUN:
    MAX_EPOCHS = 1
SEQ_LEN = {"sms": 100, "email": 400, "phone": 400}
CHANNEL_OF = {"mendeley": "sms", "uci": "sms", "uci_replication": "sms", "email_fraud": "email",
              "spamassassin": "email", "ceas_08": "email", "bothbosu": "phone"}
E2_GRID = [{"lr": lr, "hidden": h, "dropout": d, "class_weights": True}
           for lr in [1e-3, 3e-4] for h in [64, 128] for d in [0.2, 0.5]]

# Replaced by the E2 result; set BEST=lr,hidden,dropout when E2 is skipped.
BEST = {"lr": 1e-3, "hidden": 64, "dropout": 0.5, "class_weights": True}
if os.environ.get("BEST"):
    _lr, _h, _d = os.environ["BEST"].split(",")
    BEST = {"lr": float(_lr), "hidden": int(_h), "dropout": float(_d), "class_weights": True}

MENDELEY_LABELS = ["ham", "spam", "smishing"]
E1_SETS = [("mendeley", MENDELEY_LABELS, "smishing"), ("uci", ["ham", "spam"], "spam"),
           ("uci_replication", ["ham", "spam"], "spam"), ("bothbosu", ["non-scam", "scam"], "scam"),
           ("email_fraud", ["legitimate", "fraud"], "fraud"),
           ("spamassassin", ["legitimate", "spam"], "spam"), ("ceas_08", ["legitimate", "spam"], "spam")]
E3_CHANNELS = {"sms": "mendeley", "email": "email_fraud", "phone": "bothbosu"}
E3_SOURCES = [("sms",), ("email",), ("phone",), ("sms", "email"), ("sms", "phone"), ("email", "phone")]
# Start token added to every text: an empty or punctuation-only text would otherwise become a
# fully masked sequence, which the cuDNN LSTM kernel on GPU rejects.
PREFIX = "xxbos "

RES, MOD = f"{OUT}/results_bilstm", f"{OUT}/models_bilstm"
os.makedirs(RES, exist_ok=True)
os.makedirs(MOD, exist_ok=True)


# --------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------
_cache: dict[str, pd.DataFrame] = {}
SPACES_RE = re.compile(r"\s+")


def load(name: str) -> pd.DataFrame:
    """Load one processed dataset (<name>.csv or .csv.gz anywhere under DATA_DIR)."""
    if name not in _cache:
        hits = [p for p in glob.glob(f"{DATA_DIR}/**/{name}.csv*", recursive=True) if os.path.isfile(p)]
        if not hits:
            raise FileNotFoundError(f"{name}.csv(.gz) not found under {DATA_DIR}")
        df = pd.read_csv(hits[0], dtype={"label_fraud": "Int64", "label_spam": "Int64"})
        for c in ["text", "text_stripped", "template"]:
            if c in df:
                df[c] = df[c].fillna("")
        _cache[name] = df
    return _cache[name]


def dedup_key(text: str) -> str:
    """Lower-case and collapse white space (same as src/data.py)."""
    return SPACES_RE.sub(" ", text).strip().lower()


def remove_overlap(source: pd.DataFrame, target: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Drop source rows whose text or template appears in the target test set (same as src/splits.py)."""
    exact = source["text"].map(dedup_key).isin(set(target["text"].map(dedup_key)))
    template = source["template"].isin(set(target["template"]))
    counts = {"exact_text": int(exact.sum()), "template": int(template.sum()),
              "removed": int((exact | template).sum())}
    return source[~(exact | template)], counts


def label_mapper(labels: list[str]) -> Callable[[pd.Series], pd.Series]:
    """Map label names to 0..k-1 in the given order."""
    return lambda s: s.map({l: i for i, l in enumerate(labels)})


# --------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------
def build_model(train_texts: list[str], n_classes: int, seq_len: int, cfg: dict,
                seed: int = SEED) -> keras.Model:
    """BiLSTM with a trainable embedding; the vectoriser is part of the model, so it takes raw strings."""
    keras.utils.set_random_seed(seed)
    vec = layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=seq_len)
    vec.adapt(tf.data.Dataset.from_tensor_slices([PREFIX + t for t in train_texts]).batch(256))
    inp = keras.Input(shape=(), dtype="string")
    x = vec(inp)
    x = layers.Embedding(VOCAB, EMB, mask_zero=True)(x)
    x = layers.Bidirectional(layers.LSTM(cfg["hidden"]))(x)
    x = layers.Dropout(cfg["dropout"])(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(cfg["lr"]),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def as_input(texts: pd.Series) -> tf.Tensor:
    """Raw strings (with the start token) as a 1-D string tensor."""
    return tf.constant([PREFIX + t for t in texts])


def truncated_rate(texts: pd.Series, seq_len: int) -> float:
    """Share of texts with more whitespace-separated words than seq_len (approximate)."""
    return float(np.mean([len(t.split()) > seq_len for t in texts]))


# --------------------------------------------------------------------------
# Evaluation
# --------------------------------------------------------------------------
def metrics(y: np.ndarray, prob: np.ndarray, labels: list[str], pos: int) -> dict:
    """Accuracy, macro-F1, per-class scores, and recall / false-alarm rate / PR-AUC of the positive class (5.3)."""
    pred = prob.argmax(1)
    k = len(labels)
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=list(range(k)), zero_division=0)
    neg = y != pos
    return {
        "n": int(len(y)),
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro", labels=list(range(k)), zero_division=0),
        "per_class": {labels[i]: {"precision": p[i], "recall": r[i], "f1": f[i], "support": int(s[i])}
                      for i in range(k)},
        "positive_label": labels[pos],
        "positive_recall": r[pos],
        "positive_false_positive_rate": float(((pred == pos) & neg).sum() / max(neg.sum(), 1)),
        "positive_pr_auc": average_precision_score((y == pos).astype(int), prob[:, pos])
                           if 0 < (y == pos).sum() < len(y) else None,
        "confusion_matrix": confusion_matrix(y, pred, labels=list(range(k))).tolist(),
    }


def dump(obj: dict, path: str) -> None:
    """Write JSON, converting numpy scalars."""
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False,
                  default=lambda o: o.item() if hasattr(o, "item") else str(o))


def train_eval(exp: str, tag: str, train_d: pd.DataFrame, val_d: Optional[pd.DataFrame],
               tests: dict[str, pd.DataFrame], labels: list[str], pos_label: str, y_col: str,
               text_col: str, seq_len: int, cfg: dict, extra: Optional[dict] = None,
               save_dir: Optional[str] = None, y_map: Optional[Callable] = None,
               seed: int = SEED) -> list[dict]:
    """Train one model, evaluate it on val and every test set, save predictions; return summary rows."""
    extra = dict(extra or {})
    y_map = y_map or (lambda s: s.astype(int))
    if DRY_RUN:
        train_d = train_d.groupby(y_col, group_keys=False).head(DRY_RUN)
        val_d = val_d.head(DRY_RUN) if val_d is not None else None
        tests = {k: v.head(DRY_RUN) for k, v in tests.items()}
    if val_d is None or len(val_d) == 0:   # uci_replication has no val split: last 10% of train for early stopping
        cut = int(len(train_d) * 0.9)
        train_d, val_d = train_d.iloc[:cut], train_d.iloc[cut:]
        extra["early_stop_val"] = "last 10% of the training part"
    pos = labels.index(pos_label)
    ytr, yva = y_map(train_d[y_col]).values, y_map(val_d[y_col]).values

    model = build_model(train_d[text_col].tolist(), len(labels), seq_len, cfg, seed)
    counts = np.bincount(ytr, minlength=len(labels))
    cw = ({i: len(ytr) / (len(labels) * max(c, 1)) for i, c in enumerate(counts)}
          if cfg["class_weights"] else None)
    t0 = time.time()
    hist = model.fit(as_input(train_d[text_col]), ytr,
                     validation_data=(as_input(val_d[text_col]), yva),
                     epochs=MAX_EPOCHS, batch_size=BATCH, class_weight=cw, verbose=2,
                     callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                                              restore_best_weights=True)])
    train_s = time.time() - t0
    detail = {"experiment": exp, "run": tag, **extra, "config": cfg, "seq_len": seq_len, "seed": seed,
              "epochs_run": len(hist.history["loss"]),
              "best_epoch": int(np.argmin(hist.history["val_loss"])) + 1,
              "train_seconds": round(train_s, 1), "train_counts": dict(zip(labels, counts.tolist())),
              "class_weights": cw, "eval": {}}

    rows = []
    for name, d in {"val": val_d, **tests}.items():
        if d is None or len(d) == 0:
            continue
        t0 = time.time()
        prob = model.predict(as_input(d[text_col]), batch_size=256, verbose=0)
        ms = (time.time() - t0) * 1000 / len(d)
        y = y_map(d[y_col]).values
        m = metrics(y, prob, labels, pos)
        m["truncated_rate"] = truncated_rate(d[text_col], seq_len)
        m["predict_ms_per_sample"] = round(ms, 3)
        detail["eval"][name] = m
        pd.DataFrame({"sample_id": d["id"].values, "eval_set": name,
                      "true_label": [labels[i] for i in y],
                      "pred_label": [labels[i] for i in prob.argmax(1)],
                      **{f"score_{l}": prob[:, i] for i, l in enumerate(labels)}}
                     ).to_csv(f"{RES}/pred_{tag}_{name}.csv", index=False)
        row = {"experiment": exp, "run": tag, **{k: v for k, v in extra.items() if not isinstance(v, dict)},
               "eval_set": name, "n": m["n"], "macro_f1": m["macro_f1"], "accuracy": m["accuracy"],
               "positive": pos_label, "pos_recall": m["positive_recall"],
               "pos_fpr": m["positive_false_positive_rate"], "pos_pr_auc": m["positive_pr_auc"],
               "truncated_rate": m["truncated_rate"], "train_seconds": round(train_s, 1),
               "best_epoch": detail["best_epoch"]}
        rows.append(row)
        # CEAS_08 seed 0: also report without the "THE DAILY TOP 10 from CNN" campaign (README)
        if "ceas_08" in name and "seed0" in tag:
            keep = ~d["template"].str.startswith("thedailytop").values
            m2 = metrics(y[keep], prob[keep], labels, pos)
            detail["eval"][name + "_without_dailytop"] = m2
            rows.append({**row, "eval_set": name + "_without_dailytop", "n": m2["n"],
                         "macro_f1": m2["macro_f1"], "accuracy": m2["accuracy"],
                         "pos_recall": m2["positive_recall"], "pos_fpr": m2["positive_false_positive_rate"],
                         "pos_pr_auc": m2["positive_pr_auc"]})

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        model.save(f"{save_dir}/model.keras")
        dump({"labels": labels, "positive_labels": [pos_label], "seq_len": seq_len, "config": cfg,
              "text_column": text_col, "input_prefix": PREFIX, "run": tag}, f"{save_dir}/task_config.json")
    dump(detail, f"{RES}/metrics_{tag}.json")
    keras.backend.clear_session()
    return rows


summary: list[dict] = []


def record(rows: list[dict]) -> None:
    """Append rows to the running summary and rewrite bilstm_summary.csv."""
    summary.extend(rows)
    pd.DataFrame(summary).to_csv(f"{RES}/bilstm_summary.csv", index=False)
    for r in rows:
        print("[result]", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}, flush=True)


def safe(label: str, fn: Callable[[], None]) -> None:
    """Run one experiment; log any error to errors.txt and carry on with the rest."""
    try:
        fn()
    except Exception:
        msg = f"### {label}\n{traceback.format_exc()}\n"
        print(msg, flush=True)
        with open(f"{RES}/errors.txt", "a") as f:
            f.write(msg)


# --------------------------------------------------------------------------
# Experiments
# --------------------------------------------------------------------------
def run_e2() -> dict:
    """E2: tune on Mendeley split_seed0 (validation macro-F1), then one run without class weights."""
    grid: list[dict] = []

    def one(cfg: dict) -> None:
        df, sc = load("mendeley"), "split_seed0"
        tag = f"E2_lr{cfg['lr']}_h{cfg['hidden']}_d{cfg['dropout']}_cw{int(cfg['class_weights'])}"
        rows = train_eval("E2", tag, df[df[sc] == "train"], df[df[sc] == "val"], {}, MENDELEY_LABELS,
                          "smishing", "label_orig", "text", SEQ_LEN["sms"], cfg,
                          extra={"dataset": "mendeley", "seed": 0}, y_map=label_mapper(MENDELEY_LABELS))
        record(rows)
        grid.append({**cfg, "val_macro_f1": rows[0]["macro_f1"]})
        pd.DataFrame(grid).to_csv(f"{RES}/e2_tuning_val.csv", index=False)

    for cfg in E2_GRID:
        safe(f"E2 {cfg}", lambda cfg=cfg: one(cfg))
    if not grid:
        return BEST
    b = max(grid, key=lambda g: g["val_macro_f1"])
    safe("E2 no class weights", lambda: one({**{k: b[k] for k in ["lr", "hidden", "dropout"]},
                                             "class_weights": False}))
    b = max(grid, key=lambda g: g["val_macro_f1"])
    return {k: b[k] for k in ["lr", "hidden", "dropout", "class_weights"]}


def run_e1() -> None:
    """E1: every dataset with its original labels; uci_replication uses the published first-30% protocol."""
    for name, labels, pos in E1_SETS:
        for k in SEEDS:
            def one(name=name, labels=labels, pos=pos, k=k) -> None:
                df = load(name)
                df = df[df["label_orig"].isin(labels)]
                if name == "uci_replication":
                    cut = int(len(df) * 0.3)
                    tr, va, te = df.iloc[:cut], None, df.iloc[cut:]
                    tag, seed = f"E1_{name}_first30_seed{k}", SEED + k
                else:
                    sc = f"split_seed{k}"
                    tr, va, te = df[df[sc] == "train"], df[df[sc] == "val"], df[df[sc] == "test"]
                    tag, seed = f"E1_{name}_seed{k}", SEED
                record(train_eval("E1", tag, tr, va, {"test": te}, labels, pos, "label_orig", "text",
                                  SEQ_LEN[CHANNEL_OF[name]], BEST, extra={"dataset": name, "seed": k},
                                  y_map=label_mapper(labels), seed=seed,
                                  save_dir=f"{MOD}/bilstm_{name}_seed0" if k == 0 else None))
            safe(f"E1 {name} seed{k}", one)


def run_e3(size_matched: bool = False) -> None:
    """E3: fraud vs legitimate across channels; single-channel models are saved for E5.

    size_matched=True runs the E3M control instead: only the two-channel sources, with the
    training set randomly subsampled to the size of the larger of its two channels.
    """
    exp = "E3M" if size_matched else "E3"
    data = {ch: load(n)[load(n)["label_fraud"].notna()].reset_index(drop=True)
            for ch, n in E3_CHANNELS.items()}
    for k in SEEDS:
        sc = f"split_seed{k}"
        tests = {f"test_{ch}": d[d[sc] == "test"] for ch, d in data.items()}
        for src in [s for s in E3_SOURCES if len(s) == 2] if size_matched else E3_SOURCES:
            def one(src=src, k=k, sc=sc, tests=tests) -> None:
                tr_parts, va_parts, overlap = [], [], {}
                for ch in src:
                    d = data[ch]
                    tr, va = d[d[sc] == "train"], d[d[sc] == "val"]
                    for tch, td in tests.items():
                        if tch == f"test_{ch}":
                            continue   # the in-channel split is already template-grouped
                        tr, c1 = remove_overlap(tr, td)
                        va, c2 = remove_overlap(va, td)
                        overlap[f"{ch}->{tch}"] = {"train": c1, "val": c2}
                    tr_parts.append(tr)
                    va_parts.append(va)
                save = f"{MOD}/bilstm_fraud_{src[0]}_seed{k}" if len(src) == 1 else None
                extra = {"source": "+".join(src), "seed": k, "overlap_removed": overlap}
                if size_matched:
                    n = max(len(t) for t in tr_parts)
                    train = pd.concat(tr_parts, keys=list(src)).sample(n=n, random_state=k)
                    kept = train.index.get_level_values(0).value_counts()
                    extra["n_train"] = n
                    extra["size_matched"] = {"n_train": n, "per_channel": {ch: int(kept.get(ch, 0)) for ch in src}}
                else:
                    train = pd.concat(tr_parts).sample(frac=1, random_state=k)
                record(train_eval(exp, f"{exp}_{'+'.join(src)}_seed{k}", train, pd.concat(va_parts),
                                  tests, ["legitimate", "fraud"], "fraud", "label_fraud", "text",
                                  max(SEQ_LEN[ch] for ch in src), BEST, extra=extra, save_dir=save))
            safe(f"{exp} {src} seed{k}", one)


def run_e5() -> None:
    """E5 (4.4-4.5): single-channel fraud models (trained as in E3) scored on their own channel's
    test set: clean, every perturbation of perturb.py, and each again after D1 normalise()."""
    from perturb import e5_test_sets
    data = {ch: load(n)[load(n)["label_fraud"].notna()].reset_index(drop=True)
            for ch, n in E3_CHANNELS.items()}
    for k in SEEDS:
        sc = f"split_seed{k}"
        tests = {f"test_{ch}": d[d[sc] == "test"] for ch, d in data.items()}
        for ch in E3_CHANNELS:
            def one(ch=ch, k=k, sc=sc, tests=tests) -> None:
                d = data[ch]
                tr, va = d[d[sc] == "train"], d[d[sc] == "val"]
                overlap = {}
                for tch, td in tests.items():   # same training data as the E3 single-channel run
                    if tch == f"test_{ch}":
                        continue
                    tr, c1 = remove_overlap(tr, td)
                    va, c2 = remove_overlap(va, td)
                    overlap[f"{ch}->{tch}"] = {"train": c1, "val": c2}
                variants, log = e5_test_sets(tests[f"test_{ch}"], seed=0)
                log.to_csv(f"{RES}/e5_texts_{ch}_seed{k}.csv.gz", index=False)
                record(train_eval("E5", f"E5_{ch}_seed{k}", tr.sample(frac=1, random_state=k), va,
                                  {f"test_{ch}_{name}": v for name, v in variants.items()},
                                  ["legitimate", "fraud"], "fraud", "label_fraud", "text", SEQ_LEN[ch], BEST,
                                  extra={"source": ch, "seed": k, "overlap_removed": overlap}))
            safe(f"E5 {ch} seed{k}", one)


def run_e4() -> None:
    """E4: CEAS_08 <-> SpamAssassin, raw text and text_stripped (3.4)."""
    corp = {n: load(n)[load(n)["label_spam"].notna()].reset_index(drop=True)
            for n in ["ceas_08", "spamassassin"]}
    for k in SEEDS:
        sc = f"split_seed{k}"
        for src, tgt in [("spamassassin", "ceas_08"), ("ceas_08", "spamassassin")]:
            for text_col in ["text", "text_stripped"]:
                def one(src=src, tgt=tgt, text_col=text_col, k=k, sc=sc) -> None:
                    s, t = corp[src], corp[tgt]
                    tgt_test = t[t[sc] == "test"]
                    tr, c1 = remove_overlap(s[s[sc] == "train"], tgt_test)
                    va, c2 = remove_overlap(s[s[sc] == "val"], tgt_test)
                    variant = "stripped" if text_col == "text_stripped" else "raw"
                    record(train_eval("E4", f"E4_{src}_to_{tgt}_{variant}_seed{k}", tr, va,
                                      {f"test_{src}": s[s[sc] == "test"], f"test_{tgt}": tgt_test},
                                      ["legitimate", "spam"], "spam", "label_spam", text_col,
                                      SEQ_LEN["email"], BEST,
                                      extra={"source": src, "target": tgt, "text": variant, "seed": k,
                                             "overlap_removed": {"train": c1, "val": c2}}))
                safe(f"E4 {src}->{tgt} {text_col} seed{k}", one)


def write_summary() -> None:
    """Mean and std over seeds for every experiment / dataset / eval set, plus the environment."""
    s = pd.DataFrame(summary)
    if len(s):
        keys = [c for c in ["experiment", "dataset", "source", "target", "text", "eval_set"] if c in s]
        s[keys] = s[keys].fillna("-")
        agg = (s.groupby(keys)[["macro_f1", "pos_recall", "pos_fpr", "pos_pr_auc"]]
                .agg(["mean", "std", "count"]).round(4))
        agg.to_csv(f"{RES}/bilstm_mean_std.csv")
        print(agg.to_string())
    dump({"best": BEST, "vocab": VOCAB, "emb": EMB, "batch": BATCH, "max_epochs": MAX_EPOCHS,
          "seed": SEED, "seq_len": SEQ_LEN, "python": platform.python_version(),
          "tensorflow": tf.__version__, "keras": keras.__version__,
          "gpu": str(tf.config.list_physical_devices("GPU"))}, f"{RES}/env.json")


if __name__ == "__main__":
    print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"), flush=True)
    if "E2" in EXPERIMENTS:
        BEST = run_e2()
        dump(BEST, f"{RES}/e2_selected.json")
        print("[E2] selected:", BEST, flush=True)
    if "E1" in EXPERIMENTS:
        run_e1()
    if "E3" in EXPERIMENTS:
        run_e3()
    if "E4" in EXPERIMENTS:
        run_e4()
    if "E3M" in EXPERIMENTS:
        run_e3(size_matched=True)
    if "E5" in EXPERIMENTS:
        run_e5()
    write_summary()
    print("All done")


Writing bilstm_experiments.py


In [6]:
%%writefile e5_analysis.py
"""E5 robustness summary for DistilBERT and BiLSTM (EE6405 proposal v2, 4.4, 4.5 and 5.3).

Reads the E5 prediction files written by e3_e4_transfer.py (DistilBERT) and
bilstm_experiments.py (BiLSTM) and reports, per model, channel, perturbation and defence:
  - fraud recall and false alarm rate (FPR) on the changed text, next to the clean text;
  - flip rate: the share of scams detected on the clean text that the change turns into
    "legitimate" (proposal 5.3, robustness metric);
  - macro-F1.
Rows with "+D1" use the normalisation defence; "clean+D1" checks that D1 does not create
false alarms on clean text. Means and standard deviations are over the 3 split seeds.
It also draws a random sample of 50 changed scam messages for the hand check of 4.4
(readable, meaning kept).

Usage:
    python e5_analysis.py                     # RESULTS_DIR defaults to ./results

Inputs (inside RESULTS_DIR): results_e3_e4/pred_E5_*.csv and results_bilstm/pred_E5_*.csv,
and e5_texts_*.csv.gz from either folder.
Outputs: RESULTS_DIR/results_e5/ with e5_runs.csv (one row per model x channel x seed x
eval set), e5_summary.csv (mean and std over seeds) and e5_hand_check_sample.csv.
"""
from __future__ import annotations

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

# --------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------
RESULTS_DIR = Path(os.environ.get("RESULTS_DIR", "results"))
OUT = RESULTS_DIR / "results_e5"
FOLDERS = {"distilbert": "results_e3_e4", "bilstm": "results_bilstm"}
NAME_RE = re.compile(r"pred_E5_(sms|email|phone)_seed(\d)_test_\1_(.+)\.csv")
POS, NEG = "fraud", "legitimate"
N_HAND, HAND_SEED = 50, 0


# --------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------
def load_runs(res: Path) -> pd.DataFrame:
    """All E5 predictions as one long table: model, channel, seed, eval set and the labels."""
    parts = []
    for model, folder in FOLDERS.items():
        for f in sorted((res / folder).glob("pred_E5_*.csv")):
            m = NAME_RE.fullmatch(f.name)
            if not m:
                continue
            d = pd.read_csv(f, usecols=["sample_id", "true_label", "pred_label"])
            d["model"], d["channel"], d["seed"], d["eval_set"] = model, m.group(1), int(m.group(2)), m.group(3)
            parts.append(d)
    if not parts:
        raise FileNotFoundError(f"no pred_E5_*.csv files under {res}")
    return pd.concat(parts, ignore_index=True)


# --------------------------------------------------------------------------
# Evaluation
# --------------------------------------------------------------------------
def score(d: pd.DataFrame, clean: pd.DataFrame) -> dict[str, float]:
    """Recall, FPR, macro-F1 and flip rate of one eval set against the clean predictions."""
    y, p = d["true_label"].to_numpy(), d["pred_label"].to_numpy()
    pos, neg = y == POS, y == NEG
    caught = clean.set_index("sample_id")["pred_label"].eq(POS) & clean.set_index("sample_id")["true_label"].eq(POS)
    caught_ids = caught[caught].index
    now = d.set_index("sample_id").loc[caught_ids, "pred_label"]
    return {"n": len(d), "fraud_recall": float((p[pos] == POS).mean()) if pos.any() else np.nan,
            "false_alarm_rate": float((p[neg] == POS).mean()) if neg.any() else np.nan,
            "macro_f1": f1_score(y, p, labels=[NEG, POS], average="macro", zero_division=0),
            "n_caught_clean": len(caught_ids),
            "flip_rate": float((now != POS).mean()) if len(caught_ids) else np.nan}


def per_run(df: pd.DataFrame) -> pd.DataFrame:
    """Score every eval set of every model x channel x seed; flips are measured against "clean"."""
    rows = []
    for (model, ch, k), g in df.groupby(["model", "channel", "seed"]):
        clean = g[g["eval_set"] == "clean"]
        for ev, d in g.groupby("eval_set"):
            kind, _, defence = ev.partition("+")
            rows.append({"model": model, "channel": ch, "seed": k, "eval_set": ev, "perturbation": kind,
                         "defence": defence or "none", **score(d, clean)})
    return pd.DataFrame(rows)


def hand_check(res: Path) -> pd.DataFrame:
    """50 random changed scam messages (all channels and kinds, seed-0 test sets) for the hand check."""
    logs = []
    for folder in FOLDERS.values():
        for f in sorted((res / folder).glob("e5_texts_*_seed0.csv.gz")):
            ch = f.name.split("_")[2]
            d = pd.read_csv(f)
            d["channel"] = ch
            logs.append(d)
        if logs:
            break        # both models use the same changed texts; one copy is enough
    if not logs:
        return pd.DataFrame()
    d = pd.concat(logs, ignore_index=True)
    d = d[d["n_changed"] > 0]
    sample = d.sample(min(N_HAND, len(d)), random_state=HAND_SEED)
    return sample.assign(readable="", meaning_kept="", note="")[
        ["channel", "kind", "sample_id", "n_eligible", "n_changed", "original", "text", "readable",
         "meaning_kept", "note"]]


# --------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------
def main() -> None:
    """Write the per-run table, the summary over seeds and the hand-check sample."""
    runs = per_run(load_runs(RESULTS_DIR))
    OUT.mkdir(parents=True, exist_ok=True)
    runs.to_csv(OUT / "e5_runs.csv", index=False)
    keys = ["model", "channel", "perturbation", "defence"]
    cols = ["fraud_recall", "false_alarm_rate", "flip_rate", "macro_f1"]
    summary = runs.groupby(keys)[cols].agg(["mean", "std"]).round(4)
    summary.columns = [f"{a}_{b}" for a, b in summary.columns]
    summary["n_seeds"] = runs.groupby(keys).size()
    summary = summary.reset_index()
    summary.to_csv(OUT / "e5_summary.csv", index=False)
    hc = hand_check(RESULTS_DIR)
    hc.to_csv(OUT / "e5_hand_check_sample.csv", index=False)

    pd.set_option("display.width", 220)
    view = summary.pivot_table(index=["model", "channel", "perturbation"], columns="defence",
                               values="fraud_recall_mean")[["none", "D1"]].round(3)
    print("Fraud recall (mean over seeds); columns: no defence vs D1")
    print(view.to_string())
    print(f"\nwrote {len(runs)} rows to {OUT}; hand-check sample: {len(hc)} messages")


if __name__ == "__main__":
    main()


Writing e5_analysis.py


In [7]:
# Check that the three E5 datasets are attached (stops here if one is missing)
import glob, os
need = ['mendeley', 'email_fraud', 'bothbosu']
found = {n: [p for p in glob.glob(f'/kaggle/input/**/{n}.csv*', recursive=True) if os.path.isfile(p)] for n in need}
for n, p in found.items():
    print(n, '->', p[0] if p else 'MISSING')
missing = [n for n, p in found.items() if not p]
assert not missing, f'Add Input is missing {missing}: attach zyliang19/ee6405-processed and zyliang19/mendeley-sms'

mendeley -> /kaggle/input/datasets/zyliang19/mendeley-sms/mendeley.csv
email_fraud -> /kaggle/input/datasets/zyliang19/ee6405-processed/email_fraud.csv
bothbosu -> /kaggle/input/datasets/zyliang19/ee6405-processed/bothbosu.csv


In [8]:
# Unit tests for the perturbation rules (should print: 26 passed)
!python -m pytest -q test_perturb.py
!python perturb.py

..........................                                               [100%]
26 passed in 0.05s
P1             2/5  'URGEN7: your bank accoun7 has been suspended. Verify now at http://secure-bank.xyz'
P2             2/5  'URGENT: your b.a.n.k a c c o u n t has been suspended. Verify now at http://secure-bank.xyz'
P2_space       2/5  'U R G E N T: your bank a c c o u n t has been suspended. Verify now at http://secure-bank.xyz'
P2_dot         2/5  'U.R.G.E.N.T: your bank account has been suspended. V.e.r.i.f.y now at http://secure-bank.xyz'
P3             2/5  'URGENТ: your bank account has been suspende\u200bd. Verify now at http://secure-bank.xyz'
P3_lookalike   2/5  'URGENТ: your bаnk account has been suspended. Verify now at http://secure-bank.xyz'
P3_zero_width  2/5  'URGENT: your ban\u200bk account has been suspend\u200bed. Verify now at http://secure-bank.xyz'


In [9]:
# DistilBERT: 3 channels x 3 split seeds, 16 eval sets each (about 30-40 min on one T4)
!EXPERIMENTS=E5 python e3_e4_transfer.py

config.json: 100%|█████████████████████████████| 483/483 [00:00<00:00, 2.23MB/s]
tokenizer_config.json: 100%|██████████████████| 48.0/48.0 [00:00<00:00, 246kB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 13.1MB/s]
tokenizer.json: 100%|████████████████████████| 466k/466k [00:00<00:00, 36.8MB/s]
device: Tesla T4
E3 sms (mendeley): {np.int64(0): 4834, np.int64(1): 528}
E3 email (email_fraud): {np.int64(1): 4796, np.int64(0): 3159}
E3 phone (bothbosu): {np.int64(1): 799, np.int64(0): 399}
Map: 100%|████████████████████████| 3431/3431 [00:00<00:00, 15597.85 examples/s]

model.safetensors: downloading bytes: ████████████▍         |  151MB, 9.97MB/s  
model.safetensors: downloading bytes: ███████████████████▋  |  239MB, 20.9MB/s  
model.safetensors: downloading bytes: ██████████████████████|  250MB, 23.4MB/s  
model.safetensors: reconstructing file: 100%|██████|  268MB /  268MB, 25.5MB/s  
Loading weights: 100%|██████████████████████| 100/100 [00:00<00:00, 2518.24

In [10]:
# BiLSTM with the E2-selected setting (lr 1e-3, hidden 64, dropout 0.2), about 5 min
!EXPERIMENTS=E5 BEST=0.001,64,0.2 python bilstm_experiments.py

TF 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Epoch 1/10
54/54 - 7s - 122ms/step - accuracy: 0.7333 - loss: 0.3368 - val_accuracy: 0.9720 - val_loss: 0.1333
Epoch 2/10
54/54 - 1s - 15ms/step - accuracy: 0.9945 - loss: 0.0425 - val_accuracy: 0.9883 - val_loss: 0.0525
Epoch 3/10
54/54 - 1s - 14ms/step - accuracy: 0.9997 - loss: 0.0145 - val_accuracy: 0.9872 - val_loss: 0.0468
Epoch 4/10
54/54 - 1s - 14ms/step - accuracy: 0.9997 - loss: 0.0083 - val_accuracy: 0.9883 - val_loss: 0.0494
Epoch 5/10
54/54 - 1s - 14ms/step - accuracy: 0.9997 - loss: 0.0018 - val_accuracy: 0.9860 - val_loss: 0.0493
[result] {'experiment': 'E5', 'run': 'E5_sms_seed0', 'source': 'sms', 'seed': 0, 'eval_set': 'val', 'n': 858, 'macro_f1': 0.9643, 'accuracy': 0.9872, 'positive': 'fraud', 'pos_recall': np.float64(0.9524), 'pos_fpr': 0.009, 'pos_pr_auc': np.float64(0.9778), 'truncated_rate': 0.0, 'train_seconds': 9.

In [11]:
# Summary table and the 50-message hand-check sample
!RESULTS_DIR=. python e5_analysis.py

Fraud recall (mean over seeds); columns: no defence vs D1
defence                            none     D1
model      channel perturbation               
bilstm     email   P1             0.989  0.988
                   P2             0.882  0.988
                   P2_dot         0.990  0.989
                   P2_space       0.428  0.990
                   P3             0.989  0.989
                   P3_lookalike   0.988  0.989
                   P3_zero_width  0.990  0.989
                   clean          0.990  0.989
           phone   P1             1.000  1.000
                   P2             1.000  1.000
                   P2_dot         1.000  1.000
                   P2_space       1.000  1.000
                   P3             1.000  1.000
                   P3_lookalike   1.000  1.000
                   P3_zero_width  1.000  1.000
                   clean          1.000  1.000
           sms     P1             0.968  0.968
                   P2             0.823  0.968
  

In [12]:
!cat results_e3_e4/errors.txt results_bilstm/errors.txt 2>/dev/null; zip -qr e5_results.zip results_e3_e4 results_bilstm results_e5 && ls -la e5_results.zip

-rw-r--r-- 1 root root 176706659 Nov 11 12:50 e5_results.zip
